In [ ]:
!pip install pymongo
!pip install faker

In [18]:
from pymongo import MongoClient

# Conectar ao servidor MongoDB
client = MongoClient('localhost', 27017)

# Criar o banco de dados 'ecommerce'
db = client['ecommerce']

# Criar as coleções 'clientes', 'produtos' e 'pedidos'
clientes = db['clientes']
produtos = db['produtos']
pedidos = db['pedidos']

print("Banco de dados e coleções criados com sucesso!")

Banco de dados e coleções criados com sucesso!


In [ ]:
# Importações necessárias
from pymongo import MongoClient
from faker import Faker
import random
from datetime import datetime

# String de conexão com autenticação
#Lembre-se de substituir 'ecommerce_user' e 'senha123' pelos valores corretos
#Em geral, JAMAIS SUBA SCRIPTS COM SENHAS DE USUARIOS EM REPOSITORIOS PUBLICOS
# O ideal é usar o arquivo .env para armazenar as senhas e outras informações sensíveis
connection_string = "mongodb+srv:/"
client = MongoClient(connection_string)

# Teste a conexão
try:
    # Isso forçará uma conexão real para testar a autenticação
    client.server_info()
    print("Conexão estabelecida com sucesso!")
except Exception as e:
    print(f"Erro ao conectar: {e}")

# Selecionar o banco de dados
db = client.ecommerce

# Inicializar o Faker para português do Brasil
fake = Faker('pt_BR')

# Criar dados fictícios
try:
    # Inserir dados na coleção 'clientes'
    clientes = []
    for _ in range(100):
        cliente = {
            'nome': fake.name(),
            'email': fake.email(),
            'endereco': fake.address(),
            'telefone': fake.phone_number()
        }
        clientes.append(cliente)
    db.clientes.insert_many(clientes)
    print("Clientes inseridos com sucesso!")

    # Inserir dados na coleção 'produtos'
    produtos = []
    categorias = ['Eletrônicos', 'Roupas', 'Livros', 'Alimentos', 'Móveis']
    for _ in range(50):
        produto = {
            'nome': fake.word().capitalize(),
            'descricao': fake.sentence(),
            'preco': round(random.uniform(10.0, 1000.0), 2),
            'categoria': random.choice(categorias),
            'estoque': random.randint(1, 100)
        }
        produtos.append(produto)
    db.produtos.insert_many(produtos)
    print("Produtos inseridos com sucesso!")

    # Inserir dados na coleção 'pedidos'
    pedidos = []
    clientes_ids = [c['_id'] for c in db.clientes.find()]
    produtos_ids = [p['_id'] for p in db.produtos.find()]

    for _ in range(200):
        # Convertemos a data para datetime
        data_pedido = datetime.combine(fake.date_this_year(), datetime.min.time())
        pedido = {
            'cliente_id': random.choice(clientes_ids),
            'produto_id': random.choice(produtos_ids),
            'quantidade': random.randint(1, 5),
            'data_pedido': data_pedido  # Agora usando datetime
        }
        pedidos.append(pedido)
    db.pedidos.insert_many(pedidos)
    print("Pedidos inseridos com sucesso!")

except Exception as e:
    print(f"Erro ao inserir dados: {e}")

finally:
    # Fechar a conexão
    client.close()

Conexão estabelecida com sucesso!
Clientes inseridos com sucesso!
Produtos inseridos com sucesso!
Pedidos inseridos com sucesso!


In [ ]:
# Nova célula para verificação dos dados
from pymongo import MongoClient
from pprint import pprint  # Para uma visualização mais organizada

# Reconectar ao banco
connection_string = "mongodb+srv://"
client = MongoClient(connection_string)
db = client.ecommerce

# Contagem de documentos em cada coleção
print("=== Contagem de Documentos ===")
print(f"Clientes: {db.clientes.count_documents({})}")
print(f"Produtos: {db.produtos.count_documents({})}")
print(f"Pedidos: {db.pedidos.count_documents({})}")

# Exemplo de cada coleção
print("\n=== Exemplo de Cliente ===")
pprint(db.clientes.find_one())

print("\n=== Exemplo de Produto ===")
pprint(db.produtos.find_one())

print("\n=== Exemplo de Pedido ===")
pprint(db.pedidos.find_one())

# Algumas estatísticas básicas
print("\n=== Estatísticas ===")
# Produto mais caro
produto_mais_caro = db.produtos.find_one(sort=[("preco", -1)])
print(f"Produto mais caro: {produto_mais_caro['nome']} - R$ {produto_mais_caro['preco']}")

# Quantidade total de produtos em estoque
total_estoque = db.produtos.aggregate([
    {"$group": {"_id": None, "total": {"$sum": "$estoque"}}}
])
print(f"Total de produtos em estoque: {list(total_estoque)[0]['total']}")

# Quantidade total de pedidos por categoria
pedidos_por_categoria = db.produtos.aggregate([
    {"$group": {"_id": "$categoria", "count": {"$sum": 1}}}
])
print("\nProdutos por categoria:")
for cat in pedidos_por_categoria:
    print(f"{cat['_id']}: {cat['count']}")

client.close()

=== Contagem de Documentos ===
Clientes: 1300
Produtos: 850
Pedidos: 1600

=== Exemplo de Cliente ===
{'_id': ObjectId('6b0492d6423be53174015439'),
 'email': 'enzo-gabrielramos@example.net',
 'endereco': 'Estação Martins, 61\nSão Paulo\n52597-930 da Rocha / ES',
 'nome': 'Arthur Miguel Azevedo',
 'telefone': '+55 63 2081-4807'}

=== Exemplo de Produto ===
{'_id': ObjectId('6b0492d6423be5317401549d'),
 'categoria': 'Alimentos',
 'descricao': 'Vitae nesciunt eligendi aperiam possimus voluptatibus.',
 'estoque': 30,
 'nome': 'Impedit',
 'preco': 116.04}

=== Exemplo de Pedido ===
{'_id': ObjectId('6b0492d7423be531740154cf'),
 'cliente_id': ObjectId('6b0492d6423be53174015453'),
 'data_pedido': datetime.datetime(2026, 8, 30, 0, 0),
 'produto_id': ObjectId('6b0492d6423be531740154bb'),
 'quantidade': 4}

=== Estatísticas ===
Produto mais caro: Tenetur - R$ 998.83
Total de produtos em estoque: 41916

Produtos por categoria:
Alimentos: 182
Eletrônicos: 169
Livros: 149
Roupas: 171
Móveis: 179


In [ ]:
from pymongo import MongoClient
from pprint import pprint

# Conexão direta com MongoDB Atlas
connection_string = (
    "mongodb+srv:///"
    
)

try:
    with MongoClient(
        connection_string,
        serverSelectionTimeoutMS=10000
    ) as client:

        client.admin.command("ping")
        print("MongoDB Atlas conectado com sucesso!")

        db = client["ecommerce"]

        # Contagem de documentos
        print("\n=== Contagem de Documentos ===")
        print(f"Clientes: {db.clientes.count_documents({})}")
        print(f"Produtos: {db.produtos.count_documents({})}")
        print(f"Pedidos: {db.pedidos.count_documents({})}")

        # Exemplos
        print("\n=== Exemplo de Cliente ===")
        pprint(db.clientes.find_one())

        print("\n=== Exemplo de Produto ===")
        pprint(db.produtos.find_one())

        print("\n=== Exemplo de Pedido ===")
        pprint(db.pedidos.find_one())

        # Produto mais caro
        print("\n=== Produto Mais Caro ===")

        produto_mais_caro = db.produtos.find_one(
            sort=[("preco", -1)]
        )

        if produto_mais_caro:
            print(
                f"{produto_mais_caro['nome']} - "
                f"R$ {produto_mais_caro['preco']:.2f}"
            )
        else:
            print("Nenhum produto encontrado.")

        # Estoque total
        print("\n=== Total em Estoque ===")

        estoque = list(db.produtos.aggregate([
            {
                "$group": {
                    "_id": None,
                    "total": {"$sum": "$estoque"}
                }
            }
        ]))

        total_estoque = estoque[0]["total"] if estoque else 0
        print(f"Total de unidades em estoque: {total_estoque}")

        # Produtos por categoria
        print("\n=== Produtos por Categoria ===")

        categorias = list(db.produtos.aggregate([
            {
                "$group": {
                    "_id": "$categoria",
                    "total_produtos": {"$sum": 1}
                }
            },
            {
                "$sort": {"total_produtos": -1}
            }
        ]))

        total = 0

        for categoria in categorias:
            nome = categoria["_id"]
            quantidade = categoria["total_produtos"]

            print(f"{nome}: {quantidade}")
            total += quantidade

        print(f"\nTOTAL DE PRODUTOS: {total}")

except Exception as e:
    print(f"Erro ao consultar MongoDB: {e}")
    raise

print("\nVerificação concluída!")

MongoDB Atlas conectado com sucesso!

=== Contagem de Documentos ===
Clientes: 1300
Produtos: 850
Pedidos: 1600

=== Exemplo de Cliente ===
{'_id': ObjectId('6b0492d6423be53174015439'),
 'email': 'enzo-gabrielramos@example.net',
 'endereco': 'Estação Martins, 61\nSão Paulo\n52597-930 da Rocha / ES',
 'nome': 'Arthur Miguel Azevedo',
 'telefone': '+55 63 2081-4807'}

=== Exemplo de Produto ===
{'_id': ObjectId('6b0492d6423be5317401549d'),
 'categoria': 'Alimentos',
 'descricao': 'Vitae nesciunt eligendi aperiam possimus voluptatibus.',
 'estoque': 30,
 'nome': 'Impedit',
 'preco': 116.04}

=== Exemplo de Pedido ===
{'_id': ObjectId('6b0492d7423be531740154cf'),
 'cliente_id': ObjectId('6b0492d6423be53174015453'),
 'data_pedido': datetime.datetime(2026, 8, 30, 0, 0),
 'produto_id': ObjectId('6b0492d6423be531740154bb'),
 'quantidade': 4}

=== Produto Mais Caro ===
Tenetur - R$ 998.83

=== Total em Estoque ===
Total de unidades em estoque: 41916

=== Produtos por Categoria ===
Alimentos: 18